In [1]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

I0000 00:00:1779094706.286876  145405 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779094707.283259  145405 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779094710.301507  145405 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# --- 1. Configuration & Hyperparameters ---
SAMPLE_RATE           = 16000
CLIP_DURATION_MS      = 2000
WINDOW_SIZE_MS        = 30.0
WINDOW_STRIDE_MS      = 20.0
DCT_COEFFICIENT_COUNT = 40
BATCH_SIZE            = 32
EPOCHS                = 30
LEARNING_RATE         = 0.0005

DESIRED_SAMPLES        = int(SAMPLE_RATE * CLIP_DURATION_MS / 1000)
WINDOW_SIZE_SAMPLES    = int(SAMPLE_RATE * WINDOW_SIZE_MS / 1000)
WINDOW_STRIDE_SAMPLES  = int(SAMPLE_RATE * WINDOW_STRIDE_MS / 1000)
SPECTROGRAM_LENGTH     = 1 + int((DESIRED_SAMPLES - WINDOW_SIZE_SAMPLES) / WINDOW_STRIDE_SAMPLES)

print("DESIRED_SAMPLES:", DESIRED_SAMPLES)
print("SPECTROGRAM_LENGTH:", SPECTROGRAM_LENGTH)
print("DCT_COEFFICIENT_COUNT:", DCT_COEFFICIENT_COUNT)

DESIRED_SAMPLES: 32000
SPECTROGRAM_LENGTH: 99
DCT_COEFFICIENT_COUNT: 40


In [3]:
def load_and_label_npy_files():
    """Loads file paths and assigns labels based on the directory."""
    norm_files   = glob.glob(os.path.expanduser('~/npy/npy_Eng_local/Non_Distress/*.npy'))
    abnorm_files = glob.glob(os.path.expanduser('~/npy/npy_Eng_local/Distress/*.npy'))

    file_paths   = norm_files + abnorm_files
    labels       = [0] * len(norm_files) + [1] * len(abnorm_files)

    print(f"Normal files: {len(norm_files)}, Abnormal files: {len(abnorm_files)}")

    return file_paths, labels

def preprocess_audio(audio_raw):
    """
    Standardizes audio length and computes MFCCs.
    """
    # Convert to float32
    audio = tf.cast(audio_raw, tf.float32)
    # Normalize audio to [-1, 1]
    audio = audio / (tf.reduce_max(tf.abs(audio)) + 1e-6)
    # Pad or clip to DESIRED_SAMPLES
    pad_amount = tf.maximum(0, DESIRED_SAMPLES - tf.shape(audio)[0])
    audio = tf.pad(audio, [[0, pad_amount]])
    audio = audio[:DESIRED_SAMPLES]

    # STFT
    stfts = tf.signal.stft(
        audio,
        frame_length=WINDOW_SIZE_SAMPLES,
        frame_step=WINDOW_STRIDE_SAMPLES,
        fft_length=None,
        window_fn=tf.signal.hann_window
    )

    spectrograms = tf.abs(stfts)
    num_spectrogram_bins = spectrograms.shape[-1]

    # Mel Spectrogram
    linear_to_mel = tf.signal.linear_to_mel_weight_matrix(
        40,
        num_spectrogram_bins,
        SAMPLE_RATE,
        20.0,
        4000.0
    )

    mel_spectrograms = tf.tensordot(spectrograms, linear_to_mel, 1)
    # Log Mel Spectrogram
    log_mel_spectrograms = tf.math.log(mel_spectrograms + 1e-6)
    # MFCCs
    mfccs = tf.signal.mfccs_from_log_mel_spectrograms(
        log_mel_spectrograms
    )[..., :DCT_COEFFICIENT_COUNT]
    # Add channel dimension
    mfccs = tf.expand_dims(mfccs, axis=-1)
    return mfccs

def npy_generator(file_paths, labels):
    """Generator to yield preprocessed audio and labels."""
    for path, label in zip(file_paths, labels):
        try:
            data = np.load(path)
            # First column contains audio
            audio_data = data[:, 0]
            features = preprocess_audio(audio_data)
            features = features.numpy()
            yield features, label
        except Exception as e:
            print(f"Error loading {path}: {e}")
            continue

In [4]:
# --- 3. Model Definition (Inspired by your DS_CNN) ---
def get_distress_model():
    input_shape = (SPECTROGRAM_LENGTH, DCT_COEFFICIENT_COUNT, 1)
    filters     = 64

    inputs = layers.Input(shape=input_shape)

    #Initial Conv
    x = layers.Conv2D(filters, (10, 4), strides=(2, 2), padding='same',kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.2)(x)

    # Deapthwise Separable Block 1
    x = layers.DepthwiseConv2D(kernel_size=(3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (1, 1), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Deapthwise Separable Block 2
    x = layers.DepthwiseConv2D(kernel_size=(3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (1, 1), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Deapthwise Separable Block 3
    x = layers.DepthwiseConv2D(kernel_size=(3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters * 2, (1, 1), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Global Pooling and Output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    # Binary classification: use 1 unit with sigmoid OR 2 with softmax
    # Using 2 units with Softmax to match your sparse_categorical_crossentropy style
    outputs = layers.Dense(2, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=[metrics.SparseCategoricalAccuracy()]
    )
    return model

In [5]:
file_paths, labels = load_and_label_npy_files()

train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

class_weights_array = compute_class_weight(
    'balanced',
    classes=np.array([0, 1]),
    y=train_labels
)
class_weight_dict = {0: float(class_weights_array[0]), 1: float(class_weights_array[1])}
print("Class weights:", class_weight_dict)
print("Train samples:", len(train_paths))
print("Val samples:  ", len(val_paths))

Normal files: 7550, Abnormal files: 7550
Class weights: {0: 1.0, 1: 1.0}
Train samples: 12080
Val samples:   3020


In [ ]:
output_signature = (
    tf.TensorSpec(shape=(SPECTROGRAM_LENGTH, DCT_COEFFICIENT_COUNT, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32)
)

# Calculate steps per epoch for the fit method
train_steps = len(train_paths) // BATCH_SIZE
val_steps   = len(val_paths)   // BATCH_SIZE

print("train_steps:", train_steps)
print("val_steps:  ", val_steps)

train_ds = tf.data.Dataset.from_generator(
    lambda: npy_generator(train_paths, train_labels),
    output_signature=output_signature
)
train_ds = train_ds.repeat().shuffle(len(train_paths)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    lambda: npy_generator(val_paths, val_labels),
    output_signature=output_signature
)
val_ds = val_ds.repeat().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Verify shapes before training
print("\nVerifying dataset shapes...")
for feat, label in train_ds.take(1):
    print("Train batch feature shape:", feat.shape)
    print("Train batch label shape:  ", label.shape)

for feat, label in val_ds.take(1):
    print("Val batch feature shape:  ", feat.shape)
    print("Val batch label shape:    ", label.shape)

train_steps: 377
val_steps:   94

Verifying dataset shapes...


E0000 00:00:1779094713.138478  145405 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0000 00:00:1779094713.260926  145524 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1779094723.264869  145524 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 358 of 12080
I0000 00:00:1779094743.283700  145524 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 1124 of 12080
I0000 00:00:1779094753.284516  145524 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 1374 of 12080


In [ ]:
# Updated training Cell
model = get_distress_model()
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4,
        min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=8,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'model_Eng_local.h5',
        monitor='val_loss', save_best_only=True, verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

In [ ]:
model = tf.keras.models.load_model('model_Eng_local.h5')

val_results = model.evaluate(val_ds, steps=val_steps, return_dict=True)
print(f"Validation Loss:     {val_results['loss']:.4f}")
print(f"Validation Accuracy: {val_results['sparse_categorical_accuracy']:.4f}")